# Musinsa Product Review Crawler (Robust API-Based Version)
This notebook crawls reviews for a chosen product from Musinsa shopping mall using requests to communicate directly with Musinsa's internal review API, bypassing dynamic UI changes.

In [ ]:
import os
import time
import re
import urllib.request
from urllib.request import urlretrieve
from bs4 import BeautifulSoup
import pandas as pd
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By

In [ ]:
search = input('검색어를 입력하세요: ')
url = "https://www.musinsa.com/"

if not os.path.exists(search):
    os.mkdir(search)

In [ ]:
# Initialize Selenium Driver (Automatic driver management)
options = webdriver.ChromeOptions()
# options.add_argument('--headless') # Uncomment to run headlessly
browser = webdriver.Chrome(options=options)
browser.get(url)
time.sleep(2)

In [ ]:
# Enter search query
search_box = browser.find_element(By.XPATH, '//*[@class="search head-search-inp keyword-dec"]')
search_box.send_keys(search)
time.sleep(1)

In [ ]:
# Find products robustly using URL patterns rather than dynamic class names
full_html = browser.page_source
soup = BeautifulSoup(full_html, 'html.parser')

a_tags = soup.find_all('a', href=re.compile(r'/goods/\d+'))

brand_recommend = []
item_recommend = []
goods_ids = []
seen_ids = set()

for a in a_tags:
    href = a.get('href', '')
    match = re.search(r'/goods/(\d+)', href)
    if match:
        gid = match.group(1)
        if gid not in seen_ids:
            text = a.text.strip()
            if len(text) > 5:
                lines = [line.strip() for line in text.split('\n') if line.strip()]
                brand = lines[0] if len(lines) > 0 else "Brand"
                name = lines[1] if len(lines) > 1 else lines[0]
                
                brand_recommend.append(brand)
                item_recommend.append(name)
                goods_ids.append(gid)
                seen_ids.add(gid)

In [ ]:
# Select product to crawl
recommend_df = pd.DataFrame({'브랜드': brand_recommend, '상품명': item_recommend})
print(recommend_df)

search_num = int(input('원하는 상품의 번호를 입력하세요 (0부터 시작): '))
selected_goods_id = goods_ids[search_num]
selected_goods_name = item_recommend[search_num]
selected_brand = brand_recommend[search_num]

# Navigate to product details page directly
browser.get(f"https://www.musinsa.com/goods/{selected_goods_id}")
time.sleep(3)

In [ ]:
# Extract Product Details (Title & Representative Image) robustly using OpenGraph Meta Tags
full_html = browser.page_source
soup = BeautifulSoup(full_html, 'html.parser')

meta_image = soup.find('meta', property='og:image')
image_src = meta_image['content'] if meta_image else ""

meta_title = soup.find('meta', property='og:title')
if meta_title:
    result_name = meta_title['content'].strip().split(' - ')[0].strip()
else:
    result_name = selected_goods_name

result_name = re.sub(r'[\/:*?"<>|]', '_', result_name) # Clean filename

if image_src:
    if image_src.startswith('//'):
        image_src = 'https:' + image_src
    urlretrieve(image_src, f'{search}/{result_name}.jpg')
    print(f'Image saved at: {search}/{result_name}.jpg')

# Browser session is no longer needed
browser.quit()

In [ ]:
def crawl_reviews_api(goods_no, total_to_collect, has_photo="false"):
    """
    Crawls review data by hitting Musinsa's internal review API endpoint directly.
    has_photo: "true" to fetch Photo/Style reviews, "false" for general reviews.
    """
    url = "https://www.musinsa.com/api2/review/v1/view/list"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36",
        "Referer": f"https://www.musinsa.com/goods/{goods_no}"
    }
    
    score_list = []
    text_list = []
    page = 1
    count = 0
    
    print(f"[*] Crawling started for Product ID: {goods_no} (Photo mode: {has_photo})")
    
    while count < total_to_collect:
        params = {
            "goodsNo": goods_no,
            "page": page,
            "pageSize": 20,
            "sort": "up_cnt_desc",
            "myFilter": "false",
            "hasPhoto": has_photo
        }
        
        try:
            response = requests.get(url, params=params, headers=headers)
            if response.status_code != 200:
                print(f"[!] API request failed (HTTP {response.status_code}) on page {page}")
                break
                
            res_json = response.json()
            data = res_json.get("data", {})
            review_list = data.get("list", [])
            
            if not review_list:
                print(f"[+] No more reviews found at page {page}.")
                break
                
            for rev in review_list:
                content = rev.get("content", "").strip()
                score = rev.get("score", None)
                
                # Fallback attributes
                if not content:
                    content = rev.get("reviewContent", "").strip()
                if score is None:
                    score = rev.get("point", 0)
                    
                if content:
                    text_list.append(content)
                    score_list.append(score)
                    count += 1
                    
                if count >= total_to_collect:
                    break
                    
            print(f"[~] Page {page} fetched: Total collected {count}/{total_to_collect} reviews.")
            page += 1
            time.sleep(0.5) # Modest sleep to prevent IP ban
            
        except Exception as e:
            print(f"[!] Exception occured: {e}")
            break
            
    return score_list, text_list

In [ ]:
limit = int(input("수집할 리뷰 개수를 입력하세요 (예: 100): "))

print("--- Crawling General Reviews ---")
general_scores, general_texts = crawl_reviews_api(selected_goods_id, limit, has_photo="false")

print("\n--- Crawling Photo/Style Reviews ---")
photo_scores, photo_texts = crawl_reviews_api(selected_goods_id, limit, has_photo="true")

In [ ]:
def save_review_df(scores, texts, suffix):
    df = pd.DataFrame({'별점': scores, '리뷰내용': texts})
    df['리뷰내용'] = df['리뷰내용'].str.replace("\t", "", regex=False).str.replace("\n", "", regex=False)
    csv_path = f"{search}/{result_name}_{suffix}.csv"
    df.to_csv(csv_path, encoding="utf-8-sig", index=True)
    print(f"Saved CSV: {csv_path}")
    return df

df_general = save_review_df(general_scores, general_texts, 'general')
df_photo = save_review_df(photo_scores, photo_texts, 'photo')

In [ ]:
# Combine all and save final CSV
df_final = pd.DataFrame({
    '별점': general_scores + photo_scores,
    '리뷰내용': general_texts + photo_texts
})
df_final['리뷰내용'] = df_final['리뷰내용'].str.replace("\t", "", regex=False).str.replace("\n", "", regex=False)
f_csv = f"{search}/{result_name}_final.csv"
df_final.to_csv(f_csv, encoding="utf-8-sig", index=True)
print(f"Combined final CSV saved: {f_csv}")